## 1. Verificar MinIO activo

In [1]:
import subprocess
import os

print("=" * 60)
print("VERIFICACIÓN DE MINIO")
print("=" * 60)

# Verificar que MinIO está corriendo
result = subprocess.run(["pgrep", "-f", "minio"], capture_output=True, text=True)
if result.returncode == 0:
    print("Proceso MinIO encontrado")
else:
    print("MinIO no está corriendo. Ejecute ~/minio/start_minio.sh")

# Verificar puertos
for port in [9000, 9001]:
    result = subprocess.run(["nc", "-z", "localhost", str(port)], capture_output=True)
    status = "Abierto" if result.returncode == 0 else " Cerrado"
    print(f" Puerto {port}: {status}")

print("\nConfiguración S3:")
print(" Endpoint: http://localhost:9000")
print(" Access Key: minioadmin")
print(" Secret Key: minioadmin123")
print(" Bucket: spark-lake")

VERIFICACIÓN DE MINIO
Proceso MinIO encontrado
 Puerto 9000: Abierto
 Puerto 9001: Abierto

Configuración S3:
 Endpoint: http://localhost:9000
 Access Key: minioadmin
 Secret Key: minioadmin123
 Bucket: spark-lake


## 2. Verificar MinIO activo

In [2]:
from pyspark.sql import SparkSession

print("=" * 60)
print("CREACIÓN DE SPARKSESSION CON SOPORTE S3A")
print("=" * 60)

spark = (SparkSession.builder
    .appName("MinIO_DataLake")
    .master("local[*]")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1500m")

    # Configuración S3A para MinIO
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    # Delta Lake
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate())

print("SparkSession creada con soporte S3A/MinIO")
print(f" App: {spark.sparkContext.appName}")

CREACIÓN DE SPARKSESSION CON SOPORTE S3A


26/06/16 13:23:50 WARN Utils: Your hostname, david-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1; using 172.16.179.130 instead (on interface ens33)
26/06/16 13:23:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 13:23:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/16 13:23:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession creada con soporte S3A/MinIO
 App: MinIO_DataLake


## 3. Verificar Bucket con Boto3 

In [3]:
import boto3
from botocore.exceptions import ClientError

print("=" * 60)
print("VERIFICACIÓN DE BUCKET CON BOTO3")
print("=" * 60)

# Cliente S3 apuntando a MinIO
s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin123',
    region_name='us-east-1'
)

# Listar buckets
try:
    response = s3.list_buckets()
    print("Buckets disponibles:")
    for bucket in response['Buckets']:
        print(f" - {bucket['Name']} (creado: {bucket['CreationDate']})")
except ClientError as e:
    print(f" Error: {e}")

# Crear bucket si no existe
bucket_name = "spark-lake"
try:
    s3.head_bucket(Bucket=bucket_name)
    print(f"\nBucket '{bucket_name}' ya existe")
except ClientError:
    s3.create_bucket(Bucket=bucket_name)
    print(f"\nBucket '{bucket_name}' creado")

VERIFICACIÓN DE BUCKET CON BOTO3
Buckets disponibles:
 - taller1 (creado: 0001-01-01 00:00:00+00:00)

Bucket 'spark-lake' creado


## 4. Crear Datos de Prueba y Escribir a MinIO (Delta)

In [4]:
from pyspark.sql.functions import col, lit, current_date, rand, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

print("=" * 60)
print("ESCRITURA DE DELTA LAKE A MINIO")
print("=" * 60)

# Crear DataFrame de transacciones
data = [
    ("TXN001", "C001", "Laptop", 1, 2500.00, "2024-01-15"),
    ("TXN002", "C002", "Mouse", 3, 25.50, "2024-01-16"),
    ("TXN003", "C001", "Teclado", 1, 120.00, "2024-01-17"),
    ("TXN004", "C003", "Monitor", 2, 450.00, "2024-01-18"),
    ("TXN005", "C002", "Webcam", 1, 80.00, "2024-01-19"),
]

schema = StructType([
    StructField("id_transaccion", StringType(), False),
    StructField("id_cliente", StringType(), False),
    StructField("producto", StringType(), False),
    StructField("cantidad", IntegerType(), False),
    StructField("precio", DoubleType(), False),
    StructField("fecha", StringType(), False)
])

df_transacciones = spark.createDataFrame(data, schema)
df_transacciones = df_transacciones.withColumn("fecha", 
                                               col("fecha").cast(DateType()))

print("DataFrame de transacciones:")
df_transacciones.show()

# Ruta S3A
delta_path = "s3a://spark-lake/transacciones"
print(f"\nEscribiendo en: {delta_path}")
df_transacciones.write.format("delta").mode("overwrite").save(delta_path)
print("Datos escritos en Delta Lake sobre MinIO")

ESCRITURA DE DELTA LAKE A MINIO
DataFrame de transacciones:


+--------------+----------+--------+--------+------+----------+
|id_transaccion|id_cliente|producto|cantidad|precio|     fecha|
+--------------+----------+--------+--------+------+----------+
|        TXN001|      C001|  Laptop|       1|2500.0|2024-01-15|
|        TXN002|      C002|   Mouse|       3|  25.5|2024-01-16|
|        TXN003|      C001| Teclado|       1| 120.0|2024-01-17|
|        TXN004|      C003| Monitor|       2| 450.0|2024-01-18|
|        TXN005|      C002|  Webcam|       1|  80.0|2024-01-19|
+--------------+----------+--------+--------+------+----------+


Escribiendo en: s3a://spark-lake/transacciones


26/06/16 13:23:59 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Datos escritos en Delta Lake sobre MinIO


## 5. Leer desde MinIo y Transformar

In [5]:
print("=" * 60)
print("LECTURA Y TRANSFORMACIÓN DESDE MINIO")
print("=" * 60)

# Leer desde Delta en MinIO
df_delta = spark.read.format("delta").load(delta_path)
print("Datos leídos desde Delta Lake/MinIO:")
df_delta.show()

# Transformaciones
df_transformado = df_delta.withColumn(
    "total_linea", col("cantidad") * col("precio")
).withColumn(
    "categoria",
    when(col("total_linea") > 1000, "Alto Valor")
        .when(col("total_linea") > 100, "Medio Valor")
        .otherwise("Bajo Valor")
).withColumn("fecha_procesamiento", current_date())

print("\nDataFrame transformado:")
df_transformado.show()

# Escribir resultado transformado
delta_transformado = "s3a://spark-lake/transacciones_transformadas"
df_transformado.write.format("delta").mode("overwrite").save(delta_transformado)
print(f"\nDatos transformados escritos en: {delta_transformado}")

LECTURA Y TRANSFORMACIÓN DESDE MINIO
Datos leídos desde Delta Lake/MinIO:


26/06/16 13:24:05 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+--------------+----------+--------+--------+------+----------+
|id_transaccion|id_cliente|producto|cantidad|precio|     fecha|
+--------------+----------+--------+--------+------+----------+
|        TXN003|      C001| Teclado|       1| 120.0|2024-01-17|
|        TXN004|      C003| Monitor|       2| 450.0|2024-01-18|
|        TXN005|      C002|  Webcam|       1|  80.0|2024-01-19|
|        TXN001|      C001|  Laptop|       1|2500.0|2024-01-15|
|        TXN002|      C002|   Mouse|       3|  25.5|2024-01-16|
+--------------+----------+--------+--------+------+----------+


DataFrame transformado:


+--------------+----------+--------+--------+------+----------+-----------+-----------+-------------------+
|id_transaccion|id_cliente|producto|cantidad|precio|     fecha|total_linea|  categoria|fecha_procesamiento|
+--------------+----------+--------+--------+------+----------+-----------+-----------+-------------------+
|        TXN003|      C001| Teclado|       1| 120.0|2024-01-17|      120.0|Medio Valor|         2026-06-16|
|        TXN004|      C003| Monitor|       2| 450.0|2024-01-18|      900.0|Medio Valor|         2026-06-16|
|        TXN005|      C002|  Webcam|       1|  80.0|2024-01-19|       80.0| Bajo Valor|         2026-06-16|
|        TXN001|      C001|  Laptop|       1|2500.0|2024-01-15|     2500.0| Alto Valor|         2026-06-16|
|        TXN002|      C002|   Mouse|       3|  25.5|2024-01-16|       76.5| Bajo Valor|         2026-06-16|
+--------------+----------+--------+--------+------+----------+-----------+-----------+-------------------+




Datos transformados escritos en: s3a://spark-lake/transacciones_transformadas


## 6. Time Travel con Delta Lake en MinIO

In [6]:
from delta import DeltaTable

print("=" * 60)
print("TIME TRAVEL CON DELTA LAKE")
print("=" * 60)

# Ver historial
delta_table = DeltaTable.forPath(spark, delta_path)
print("Historial de versiones:")
delta_table.history().show(truncate=False)

# Agregar nuevos datos (nueva versión)
nuevas_transacciones = [
    ("TXN006", "C004", "Audífonos", 2, 60.00, "2024-01-20"),
    ("TXN007", "C005", "Disco Duro", 1, 150.00, "2024-01-21"),
]

df_nuevas = spark.createDataFrame(nuevas_transacciones, schema)
df_nuevas = df_nuevas.withColumn("fecha", col("fecha").cast(DateType()))

# Merge (UPSERT) - actualizar si existe, insertar si no
delta_table.alias("target").merge(
    df_nuevas.alias("source"), "target.id_transaccion = source.id_transaccion"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print("Merge ejecutado (UPSERT)")

# Ver nuevo historial
print("\nNuevo historial:")
delta_table.history().show(truncate=False)

# Leer versión anterior
df_version_0 = spark.read.format("delta").option("versionAsOf", 0).load(delta_path)
print("\nVersión 0 (original):")

df_version_0.show()
df_version_1 = spark.read.format("delta").option("versionAsOf",
1).load(delta_path)

print("\nVersión 1 (después del merge):")
df_version_1.show()

TIME TRAVEL CON DELTA LAKE
Historial de versiones:
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                   |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+--------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|0      |2026-06-16 13:24:04|NULL  |NULL    |WRITE    |{mode -> Overwrite, partitionBy -> []}|NULL|NULL    |NULL     |NULL       |Serializable

Merge ejecutado (UPSERT)

Nuevo historial:
+-------+-------------------+------+--------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------

+--------------+----------+--------+--------+------+----------+
|id_transaccion|id_cliente|producto|cantidad|precio|     fecha|
+--------------+----------+--------+--------+------+----------+
|        TXN003|      C001| Teclado|       1| 120.0|2024-01-17|
|        TXN004|      C003| Monitor|       2| 450.0|2024-01-18|
|        TXN005|      C002|  Webcam|       1|  80.0|2024-01-19|
|        TXN001|      C001|  Laptop|       1|2500.0|2024-01-15|
|        TXN002|      C002|   Mouse|       3|  25.5|2024-01-16|
+--------------+----------+--------+--------+------+----------+


Versión 1 (después del merge):


+--------------+----------+----------+--------+------+----------+
|id_transaccion|id_cliente|  producto|cantidad|precio|     fecha|
+--------------+----------+----------+--------+------+----------+
|        TXN003|      C001|   Teclado|       1| 120.0|2024-01-17|
|        TXN004|      C003|   Monitor|       2| 450.0|2024-01-18|
|        TXN005|      C002|    Webcam|       1|  80.0|2024-01-19|
|        TXN006|      C004| Audífonos|       2|  60.0|2024-01-20|
|        TXN007|      C005|Disco Duro|       1| 150.0|2024-01-21|
|        TXN001|      C001|    Laptop|       1|2500.0|2024-01-15|
|        TXN002|      C002|     Mouse|       3|  25.5|2024-01-16|
+--------------+----------+----------+--------+------+----------+



## 7. Pipeline Completo: MinIO → Spark → PostgreSQL

In [7]:
print("=" * 60)
print("PIPELINE END-TO-END: MINIO → SPARK → POSTGRESQL")
print("=" * 60)

# Leer datos transformados desde MinIO
df_final = spark.read.format("delta").load(delta_transformado)

# Configuración JDBC (misma del Notebook 03)
jdbc_url = "jdbc:postgresql://localhost:5432/electiva"
connection_properties = {
    "user": "postgres",
    "password": "123abc",
    "driver": "org.postgresql.Driver"
}

# Escribir a PostgreSQL
df_final.write.jdbc(
    url=jdbc_url,
    table="transacciones_procesadas",
    mode="overwrite",
    properties=connection_properties
)

print("Datos de MinIO escritos en PostgreSQL (tabla: transacciones_procesadas)")

# Verificar lectura desde PostgreSQL
df_verificacion = spark.read.jdbc(
    url=jdbc_url,
    table="transacciones_procesadas",
    properties=connection_properties
)

print("\nVerificación - Datos en PostgreSQL:")
df_verificacion.show()

# Conteos cruzados
count_minio = df_final.count()
count_pg = df_verificacion.count()
print(f"\nConteo en MinIO: {count_minio}")
print(f"Conteo en PostgreSQL: {count_pg}")
print(f"Integridad verificada: {'CORRECTA' if count_minio == count_pg else 'ERROR'}")

PIPELINE END-TO-END: MINIO → SPARK → POSTGRESQL


Datos de MinIO escritos en PostgreSQL (tabla: transacciones_procesadas)

Verificación - Datos en PostgreSQL:
+--------------+----------+--------+--------+------+----------+-----------+-----------+-------------------+
|id_transaccion|id_cliente|producto|cantidad|precio|     fecha|total_linea|  categoria|fecha_procesamiento|
+--------------+----------+--------+--------+------+----------+-----------+-----------+-------------------+
|        TXN003|      C001| Teclado|       1| 120.0|2024-01-17|      120.0|Medio Valor|         2026-06-16|
|        TXN004|      C003| Monitor|       2| 450.0|2024-01-18|      900.0|Medio Valor|         2026-06-16|
|        TXN005|      C002|  Webcam|       1|  80.0|2024-01-19|       80.0| Bajo Valor|         2026-06-16|
|        TXN001|      C001|  Laptop|       1|2500.0|2024-01-15|     2500.0| Alto Valor|         2026-06-16|
|        TXN002|      C002|   Mouse|       3|  25.5|2024-01-16|       76.5| Bajo Valor|         2026-06-16|
+--------------+----------+

[Stage 68:====================================>                   (33 + 2) / 50]


Conteo en MinIO: 5
Conteo en PostgreSQL: 5
Integridad verificada: CORRECTA


## 8. Detener Spark

In [8]:
spark.stop()
print("\n NOTEBOOK 04 COMPLETADO")


 NOTEBOOK 04 COMPLETADO
